# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display dataset title and description
print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Tip:** All entities are referenced by their `@id` as per Croissant specification.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    if 'field' in rs:
        if isinstance(rs['field'], list):
            print('  Fields:')
            for field in rs['field']:
                print(f"    - {field['@id']}")
        else:
            print(f"  Field: {rs['field']['@id']}")
    else:
        print('  No fields listed in this record set.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Choose a record set to explore; for demonstration use the first record set available.
record_sets = dataset.metadata.record_sets
if len(record_sets) == 0:
    raise ValueError("No record sets found in this dataset.")

# Use the first record set for demonstration purposes
main_record_set = record_sets[0]['@id']
print(f"Selected Record Set: {main_record_set}")

# If you wish to process multiple record sets, collect their @ids into a list
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Extract each record set as a DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
    else:
        print(f"Warning: No records found for record set {rs_id}")

# Display columns and head for the selected main record set
if main_record_set in dataframes:
    print(f"Columns in {main_record_set}:")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())
else:
    print(f"No data loaded for record set {main_record_set}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming distributions, or grouping by key attributes.

**All column and field references are by their `@id` name.**

In [ ]:
# Pick a numeric field/column for demonstration
# List the columns so user can select one
df = dataframes[main_record_set]
print('Available columns:', df.columns.tolist())

# Example: Let's try to pick columns commonly representing numeric fields (e.g., abundance, MW, coverage, etc.)
import numpy as np

# Try to auto-detect some numeric column to demonstrate; you can manually override 'numeric_field' below.
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
else:
    # Try to coerce some typical fields to numeric if types are object
    possible_numeric = [
        col for col in df.columns if any(key in col.lower() for key in ['abundance', 'coverage', 'mw', 'count', 'number'])
    ]
    for col in possible_numeric:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
if numeric_field is None:
    raise ValueError('No numeric field found automatically. Please inspect df.columns and choose a numeric field to proceed.')

print(f"Selected numeric field for filtering: {numeric_field}")
threshold = np.nanmedian(df[numeric_field])  # Use median as a practical threshold

# Filter records
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize
if filtered_df[numeric_field].std() > 0:
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
else:
    filtered_df[f"{numeric_field}_normalized"] = 0  # constant field
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Choose a grouping field (e.g. protein_description, or similar), try auto-detect non-numeric
non_numeric_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
group_field = None
if len(non_numeric_fields) > 0:
    for key in ['description', 'category', 'sample', 'id']:
        for col in non_numeric_fields:
            if key in col.lower():
                group_field = col
                break
        if group_field:
            break
    if not group_field:
        group_field = non_numeric_fields[0]

if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())
else:
    print(f"No suitable group field found. Available candidates: {non_numeric_fields}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of the numeric field, normalized if available
plt.figure(figsize=(8,4))
filtered_df[numeric_field].hist(bins=40, edgecolor='black')
plt.title(f'Distribution of {numeric_field} (> median)')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If there is a group_field, plot group means
if 'grouped_df' in locals() and group_field in filtered_df.columns:
    grouped_df[numeric_field].sort_values(ascending=False).head(10).plot(kind='bar')
    plt.title(f'Top 10 {group_field} Mean {numeric_field}')
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset features protein abundance and modifications in human mast cell extracellular vesicles, annotated per UniProt.
- We examined its structure via record sets and fields per Croissant `@id` references.
- Filtered and normalized a representative numeric field, demonstrating grouping operations when possible.
- Initial visualizations revealed distribution and group-level summaries, highlighting the potential of `mlcroissant` for FAIR biomedical data exploration.